In [1]:

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:

load_dotenv()

model = ChatOpenAI()

In [2]:

class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [ ]:
def create_outline(state: BlogState) -> BlogState:
    
    # fetch title from the state
    title = state['title']
    
    # call llm gen outline 
    prompt  = f"Create a detailed outline for a blog post with the title: {title}"
    outline = model.invoke(prompt).content
    
    # update state
    state['outline'] = outline
    
    return state
    

In [ ]:
# function for creating a full blog

def create_blog(state: BlogState) -> BlogState:
    
    title = state['title']
    outline = state['outline']
    
    prompt = f"Write a blog post with the title: {title} and the following outline: {outline}"
    content = model.invoke(prompt).content
    
    state['content'] = content
    
    return state

In [ ]:
graph = StateGraph(BlogState)

# nodes 
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)


# edges 
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()


In [ ]:
initial_state = {'title': "The Future of AI in Healthcare"}

final_state = workflow.invoke(initial_state)

print(final_state['content'])

In [ ]:
print(final_state['outline'])